In [2]:
import wikipedia

# Wikipedia strictly enforces: <AppName>/<Version> (<Contact Information>)
wikipedia.set_user_agent("LangChainResearchBot/1.0 (contact: test@example.com)")

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\user\AppData\Local\Temp\ipykernel_9476\1921846137.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [3]:
"""api_wrapper_wiki= WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=400)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name"""

'api_wrapper_wiki= WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=400)\nwiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)\nwiki.name'

In [4]:
class SafeWikipediaAPIWrapper(WikipediaAPIWrapper):
    def run(self, query: str) -> str:
        try:
            return super().run(query)
        except Exception as e:
            return f"Wikipedia tool error: {e}. Continue without this article."

api_wrapper_wiki = SafeWikipediaAPIWrapper(
    top_k_results=2, 
    doc_content_chars_max=200
)

wiki = WikipediaQueryRun(
    api_wrapper=api_wrapper_wiki,
    handle_tool_error=True
)

In [5]:
"""import arxiv

# Monkey-patch older API behavior onto arxiv.Search
if not hasattr(arxiv.Search, "results"):
    arxiv.Search.results = lambda self: arxiv.Client().results(self)

from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=400)
arxiv_tool = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)"""

'import arxiv\n\n# Monkey-patch older API behavior onto arxiv.Search\nif not hasattr(arxiv.Search, "results"):\n    arxiv.Search.results = lambda self: arxiv.Client().results(self)\n\nfrom langchain_community.tools import ArxivQueryRun\nfrom langchain_community.utilities import ArxivAPIWrapper\n\napi_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=400)\narxiv_tool = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)'

In [6]:
import os
os.environ["USER_AGENT"] = "ResearchAgent/1.0 (contact: your_email@example.com)"

## custom tools[rag tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

SSLError: HTTPSConnectionPool(host='docs.smith.langchain.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLError(1, '[SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:1006)')))

In [ ]:
# NEW: Initialize the open-source embedding model
# This downloads a lightweight, fast model to run entirely on your local machine
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Build vector store using the local embeddings
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026D349D1D90>, search_kwargs={})

In [ ]:
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [ ]:
# If you used 'import arxiv':
tools = [wiki, retriever_tool]
tools

[WikipediaQueryRun(handle_tool_error=True, api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Search Engine\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=2, lang='en', load_all_available_meta=False, doc_content_chars_max=400)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000026D3499B6A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000026D38105440>)]

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env into environment
load_dotenv()

# Access the key
#google_api_key = os.getenv("GEMINI_API_KEY")
hf_api_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")
groq_api_key = os.getenv("GROQ_API_KEY")


In [ ]:
"""from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.environ.get("GEMINI_API_KEY"),
    temperature=0
)

# Test invocation
response = model.invoke("Explain vector databases in one sentence.")
print(response.content)"""

'from langchain_google_genai import ChatGoogleGenerativeAI\n\nmodel = ChatGoogleGenerativeAI(\n    model="gemini-3.6-flash",\n    google_api_key=os.environ.get("GEMINI_API_KEY"),\n    temperature=0\n)\n\n# Test invocation\nresponse = model.invoke("Explain vector databases in one sentence.")\nprint(response.content)'

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
import langchain
print(langchain.__version__)

1.4.0


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful research assistant. "
        "Use Wikipedia for general factual questions. "
        "Use arXiv for scientific research questions. "
        "Use the LangSmith search tool when the question is about LangSmith."
    )
)

In [ ]:
inputs = {
    "messages": [("user", "tell me about langsmith?")]
}

for token, metadata in agent.stream(inputs, stream_mode="messages"):
    # 1. Stream the model's text response live
    if metadata.get("langgraph_node") == "model" and token.content:
        print(token.content, end="", flush=True)
        
    # 2. Optionally detect when the model decides to invoke a tool
    if hasattr(token, "tool_call_chunks") and token.tool_call_chunks:
        for chunk in token.tool_call_chunks:
            if chunk.get("name"):
                print(f"\n⚡ Using Tool: {chunk['name']}...\n")


⚡ Using Tool: langsmith-search...

**LangSmith – An Observability Platform for LLM‑Powered Applications**

| Aspect | What It Is / Does |
|--------|-------------------|
| **Core purpose** | Provides end‑to‑end observability, debugging, evaluation, and prompt‑engineering tooling for applications that use large language models (LLMs). |
| **Key concepts** | • **Traces** – a record of every step an LLM‑agent takes in production (inputs, prompts, model calls, outputs, metadata). <br>• **Observability** – dashboards and metrics that let you monitor latency, error rates, token usage, and quality across the whole fleet of runs. <br>• **Evaluation** – built‑in pipelines to run automated quality checks (e.g., correctness, relevance, safety) against a reference dataset. <br>• **Prompt engineering** – UI for iterating on prompts, versioning them, and seeing how changes affect downstream performance. |
| **Main features** | 1. **Trace collection & UI** – every LLM call is automatically logged; yo

In [ ]:
"""result = agent.invoke({
    "messages": [
        ("user", "tell me about michael jackson and then search for me about langsmith ?")
    ]
})

print(result["messages"][-1].content)"""

SyntaxError: invalid syntax (356991372.py, line 1)

In [ ]:
for message in result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for call in message.tool_calls:
            print("TOOL USED:", call["name"])

TOOL USED: wikipedia
TOOL USED: langsmith-search
